In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["ABSL_MIN_LOG_LEVEL"] = "3"
import warnings
import datetime
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from keras.layers import Dense, Conv2D, MaxPool2D,Flatten, Input,Dropout
import seaborn as sns
import visualkeras
from visualkeras.lenet import lenet_view
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix as sk_confusion_matrix, ConfusionMatrixDisplay , classification_report ,accuracy_score, precision_score, recall_score, f1_score , roc_curve , auc, roc_auc_score
from pathlib import Path
from tensorflow.keras import layers
from tensorflow.keras.utils import plot_model
from sklearn.preprocessing import label_binarize
import logging
import collections

logger= logging.getLogger(__name__)


warnings.filterwarnings(
    "ignore",
    message="The legend_text_spacing_offset parameter is deprecated.*",
    category=UserWarning,
    module="visualkeras.layered"
)


In [ ]:
df_train= pd.read_csv('../data/sign_mnist_train.csv')
df_test= pd.read_csv('../data/sign_mnist_test.csv')

In [ ]:
# funcion para carga dataset
def load_data(df: pd.DataFrame ):
    """ Separa las caracteristicas (pixeles) y las etiquetas (labels)
     del conjunto de datos a partir de un DataFrame cargado previamente.
    Args:
        df (pd.DataFrame): DataFrame que contiene la columna label y las columnas correspondientes
        a los píxeles de las imagenes.
    Returns:
        tuple: (X, y)
        - (X) arreglo de los valores píxeles
        - (y) arreglos de las etiquetas de las clases.
    """    
    y= df['label'].values
    X= df.drop(columns=['label']).values
    return X, y

In [ ]:
X_train, y_train = load_data(df_train)
X_test, y_test= load_data(df_test)

In [ ]:
print(f'Forma de X_train: {X_train.shape} - Forma de y_train: {y_train.shape}')
print(f'Forma de X_test: {X_test.shape} - Forma de y_test: {y_test.shape}')

In [ ]:
EPOCHS=30
BATCH_SIZE=64
IMG_HEIGHT,IMG_WIDTH= 28, 28

In [ ]:
# cargar datos del dataset real.
ruta_real_img= '../dataReal/'
try:
    if not os.path.exists(ruta_real_img):
        raise FileNotFoundError(f'La ruta especificada no existe: {os.path.abspath(ruta_real_img)}')

    logger.info(f'Iniciando la carga del dataset desde: {ruta_real_img}')

    img_ruta_real= tf.keras.utils.image_dataset_from_directory(
    ruta_real_img,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    label_mode='categorical'
    )
    # se guardan las clases antes de mapear
    class_names= img_ruta_real.class_names
    print(f'clases detectadas: {class_names}')
    logger.info('Data set real cargado y estructurado correctamente')
except Exception as e:
    logger.error(f'Error critico al cargar el dataset {e}')

In [ ]:
# normalizacion
def normalization(x, y):
    """ Toma los valores de los píxeles de las imágenes (en escala de 0 a 255)

  y aplica una transformación de escalado de características dividiendo
  por 255.0 para llevarlos al rango [0.0, 1.0], convirtiéndolos a float32.

  Args:
      X (tf.Tensor): Tensor que contiene los lotes de imágenes a
        normalizar.
      Y (tf.Tensor): Tensor que contiene las etiquetas asociadas a las
        imágenes.

  Returns:
      tuple: Una tupla con (images_normalizadas, etiquetas) listas para el
      pipeline.
  """
    x= tf.cast(x, tf.float32)
    x= tf.cond(tf.reduce_max(x) > 1.0, lambda: x /255.0, lambda:x)
    return x, y

In [ ]:
# Aplicar tu función de normalización y optimización
img_ruta_real = img_ruta_real.map(
    normalization, num_parallel_calls=tf.data.AUTOTUNE
)
img_ruta_real = img_ruta_real.prefetch(tf.data.AUTOTUNE)

In [ ]:
# funcion split train and test
def split(X_data: np.ndarray, y_data: np.ndarray, test_size=0.2):
    """ Divide el conjunto de datos de entrenamiento y validación 
    de forma estratificada y reproducibles mediante semillas aleatorias.

    Args:
        X_data (np.ndarray): Arreglo con caracteristicas de píxeles de las imágenes.
        y_data (np.ndarray): Arreglo de la etiquetas de las clases.
        de manera opcional se opta el parametro test_size , proporcion del dataset destinado
        a validacion, por defecto se utilizo el 20%.

    Returns:
        tupla: devuelve una tupla que contiene, X_train, x_val, y_train, y_val.
    """    
    np.random.seed(42)
    tf.keras.utils.set_random_seed(42)
    X_train, x_val, y_train, y_val= train_test_split(
        X_data,y_data,
        test_size=test_size,
        random_state= 42,
        stratify=y_data,
    )
    return X_train, x_val, y_train, y_val

In [ ]:
X_train, x_val, y_train, y_val= split(X_train, y_train)

In [ ]:
# funcion de redimension
def redim(X: np.ndarray)-> np.ndarray:
    """ Transforma matriz aplanada (-1, 784) a un tensor 4D (-1,28,28,1) y lo convierte en float32.
    Args:
        X (np.ndarray): Matriz de 2D con los datos aplanados (shape (-1, 784)).
    Returns:
        np.ndarray: Retorna el tensor transformado en 4D.
    """    
    return X.reshape(-1,28,28,1).astype('float32')

In [ ]:
# transformacion
X_train_res= redim(X_train)
x_val_res= redim(x_val)
X_test_res= redim(X_test)

In [ ]:
# validamos la imagen del dataset
plt.figure(figsize=(6,5))
plt.imshow(X_train_res[0], cmap='gray')
plt.show()

In [ ]:
def build_dataset(X: tf.Tensor | np.ndarray, y: tf.Tensor | np.ndarray,
                batch_size:int = BATCH_SIZE, shuffle_buffer:int |None = None,
                map_fn=normalization, prefetch_buffer:int | None = tf.data.AUTOTUNE,)-> tf.data.Dataset:
    """ Construye un pipeline configurable para train, validacion y test.
        Aplica de forma opcional el mezclado aleatorio por buffer, agrupa las muestras
        en lotes, transforma cada elemento mediante una función de mapeo (como la
        normalización de píxeles) y habilita la precarga asíncrona para optimizar el
        rendimiento del flujo de datos.
    Args:
        X (tf.Tensor | np.array): Arreglo o tensor de las caracteristicas.
        y (tf.Tensor | np.array): Arreglo o tensor de las etiquetas.
        bacth_size (int, optional): Número de muestra por lotes en el dataset.
        shuffle_buffer (int | None, optional): Tamano del buffer para mezclar de forma aleatoria.
        map_fn (_type_, optional): Funcion aplicada a cada elemento del dataset, en nuestro caso la función de normalización.
        prefetch_buffer (int, optional): Tamano del buffer de precarga o  tf.data.AUTOTUNE para ajuste automatico si es None se desactiva.

    Returns:
        tf.data.Dataset: Objeto configurado y listo para el entrenamiento, validación y evaluación.
    """    
    ds= tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle_buffer is not None:
        ds= ds.shuffle(buffer_size=shuffle_buffer, seed=42)
        ds= ds.batch(batch_size)

    if map_fn is not None:
        ds= ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)

    if prefetch_buffer is not None:
        ds= ds.prefetch(prefetch_buffer)

    return ds

In [ ]:
def build_dataset(
    X: tf.Tensor | np.ndarray,
    y: tf.Tensor | np.ndarray,
    batch_size: int = BATCH_SIZE,
    shuffle_buffer: int | None = None,
    map_fn=normalization,
    prefetch_buffer: int | None = tf.data.AUTOTUNE,
) -> tf.data.Dataset:
  """ Construye un pipeline configurable para train, validacion y test.
                  Aplica de forma opcional el mezclado aleatorio por buffer, agrupa las muestras
                  en lotes, transforma cada elemento mediante una función de mapeo (como la
                  normalización de píxeles) y habilita la precarga asíncrona para optimizar el
                  rendimiento del flujo de datos.
              Args:
                  X (tf.Tensor | np.array): Arreglo o tensor de las caracteristicas.
                  y (tf.Tensor | np.array): Arreglo o tensor de las etiquetas.
                  bacth_size (int, optional): Número de muestra por lotes en el dataset.
                  shuffle_buffer (int | None, optional): Tamano del buffer para mezclar de forma aleatoria.
                  map_fn (_type_, optional): Funcion aplicada a cada elemento del dataset, en nuestro caso la función de normalización.
                  prefetch_buffer (int, optional): Tamano del buffer de precarga o  tf.data.AUTOTUNE para ajuste automatico si es None se desactiva.
          
              Returns:
                  tf.data.Dataset: Objeto configurado y listo para el entrenamiento, validación y evaluación.
      """ 
  X = np.asarray(X, dtype=np.float32)
  if X.ndim == 2 and X.shape[1] == 784:
    X = X.reshape(-1, 28, 28, 1)
  elif X.ndim == 3:
    X = X[..., np.newaxis]  # (N, 28, 28) -> (N, 28, 28, 1)

  ds = tf.data.Dataset.from_tensor_slices((X, y))

  if shuffle_buffer is not None and shuffle_buffer > 0:
    ds = ds.shuffle(buffer_size=shuffle_buffer, seed=42)

  if map_fn is not None:
    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)

  ds = ds.batch(batch_size)

  if prefetch_buffer is not None:
    ds = ds.prefetch(prefetch_buffer)

  return ds

In [ ]:
train_dataset= build_dataset(
    X_train_res, y_train,
    batch_size=BATCH_SIZE,
    shuffle_buffer=10000,
    map_fn=normalization,
    prefetch_buffer= tf.data.AUTOTUNE
)
val_dataset= build_dataset(
    x_val_res, y_val,
    batch_size=BATCH_SIZE,
    map_fn=normalization,
    prefetch_buffer=tf.data.AUTOTUNE
)
test_dataset= build_dataset(
    X_test_res, y_test,
    batch_size=BATCH_SIZE,
    map_fn=normalization
)

In [ ]:
# distribucion de clases train original
s_train = pd.Series(y_train).value_counts().sort_index()

# Distribución en test original
s_test = pd.Series(y_test).value_counts().sort_index()

df_dist = pd.DataFrame({'Train original': s_train, 'Test original': s_test})
print(df_dist)

df_dist.plot(kind='bar', figsize=(11, 4))
plt.title('Distribución de clases - Datasets originales (Sign MNIST)', fontweight='bold')
plt.xlabel('Clases',fontweight='bold')
plt.ylabel('Cantidad de muestras', fontweight='bold')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Build Neuronal networks CNN Complex and Percetrón Multicapa MLP

In [ ]:
def build_model_mlp(input_shape=(28,28,1),num_class:int=25,  model_name: str = 'Mi_red')-> tf.keras.Model:
    """Construye un modelo Perceptrón Multicapa (MLP) secuencial para clasificacion.
    Args:
        input_shape (tuple, optional): Forma de los datos de entrada por defecto (28,28,1).
        num_class (int): numero de clases de salida, por defecto 25.
        model_name (str, optional): proporciona el nombre del modelo, por defecto Mi_red.

    Returns:
        tf.keras.Model: Instancia de un modelo secuencial de keras con capas densas listo para compilar. 
    """    
    model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=input_shape, name='Input_layer_Dense'),
    tf.keras.layers.Flatten(name='flatten'),
    tf.keras.layers.Dense(128, activation='relu', name='layer_Dense_intermediate'),
    tf.keras.layers.Dense(num_class, activation='softmax', name='Output_layer')], name=model_name)
    return model

In [ ]:
model_mlp= build_model_mlp(model_name='red_baseline_mlp')

In [ ]:

visualkeras.layered_view(model_mlp, legend=True, draw_volume=True, background_fill='black', font_color='white')


In [ ]:
visualkeras.lenet_view(model_mlp)

In [ ]:
visualkeras.graph_view(model_mlp, background_fill='black', image_fit='cover', layer_spacing= 120, padding=0.5)

# Red_C
- la siguiente red contendra 3 tipos de tecnicas de regularizacion, data augmentation que son las siguientes; Rotation, Escalado que corresponde a las transformaciones geometricas, la otra tecnica que se aplicara es aumento del espacio de color; contraste.

- rotation, parametros aplicados, factor 0.03 , randomzoom height=-0.05, 0.05, widhtfactor=-0.05, 0.05
- batch normalization, lo que hace es que toma las activaciones (salidas numericas) de una capa y las normaliza, cada entrada donde le ajusta la media y la varianza, en tiempo real para cada lote antes de pasarlas a la siguiente capa.
- Inverse Time decay, permite estabilizar la perdida de validación. es decir que a medida que transcurre el entrenamiento , la tasa de aprendisaje reduce progresivamente el tamano del optimizador.
- he_normal, manipulación de los pesos, indicamos a tensorflow que los pesos de esa capa densa no deben comenzar en cero, ni con valores al azar altos, que evitamos al utlizar he_normal, evitamos el desvanecimiento o explosión de gradientes, si inicializamos los pesos muy altos las activaciones se saturan y el modelo no aprende y si lo acercamos a una valor casi cero la senal se apaga. son pesos aleatorios pero no al lote , son aleatorios inteligentes para que la red empiece a aprender desde la primera epoca sin estancarse.

In [ ]:
def build_model_cnn(n_filters:np.ndarray, input_shape=(28,28,1),num_class:int=25,  model_name='model_cnn' )->tf.keras.Model:
    """Construye la arquitectura de una cnn con Data Augmentation. 
    Args:
        n_filters (np.ndarray): Lista o arreglo con el numero de filtros para las capas convolucionales.
        input_shape (tuple, optional): Forma de los datos de entrada, por defecto (28,28,1).
        num_class (int, optional): Número de clases de salida, por defecto 25 clases.
        model_name (str, optional): Indica el nombre del modelo, por defecto model_cnn.

    Returns:
        tf.keras.Model: Instancia de un modelo secuencial de keras con Data Augmentation.
    """    
    model= tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=input_shape, name='Input_layer'),
        tf.keras.layers.RandomRotation(factor=0.03, name='aug_rotation'),
        tf.keras.layers.RandomZoom(height_factor=(-0.05,0.05), width_factor=(-0.05,0.05), name='aug_zoom'),
        tf.keras.layers.RandomContrast(factor=0.1, name='aug_contrast'),
        tf.keras.layers.Conv2D(filters=n_filters[0],kernel_size=(3,3), activation='relu', name='layer_conv_1'),
        tf.keras.layers.BatchNormalization(name='batch_normalization'),
        tf.keras.layers.MaxPool2D((2,2), name='Maxpool_1'),
        tf.keras.layers.Flatten(name='flatten'),
        tf.keras.layers.Dropout(0.2, name='layer_Dropout_0.2'),
        tf.keras.layers.Dense(128, activation='relu', kernel_initializer='he_normal', name='layer_Dense_1'),
        tf.keras.layers.Dense(num_class, activation='softmax', name='layer_output'),
    ], name=model_name)
    return model

In [ ]:
complex_model= build_model_cnn([64],model_name='modelo_3')

In [ ]:
visualkeras.layered_view(complex_model, legend=True, draw_volume=True, background_fill='black', font_color='white')

In [ ]:
visualkeras.lenet_view(complex_model)

In [ ]:
visualkeras.graph_view(complex_model,background_fill='black', image_fit='cover', layer_spacing=150, padding= 0.5 )

In [ ]:
# callbacks earlyStopping, TensorBoard and model list
model=[model_mlp, complex_model]
callbacks_TensorBoard={}
for mod in model:
    log_dir= f'log/{mod.name}_{datetime.datetime.now().strftime("%d-%m-%y,%H-%M-%S")}'
    callbacks_ear_stopp= tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    callbacks_TensorBoard[mod.name]= tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

- En esta fase se realiza una segunda experiencia de entrenamiento, incorporando inverse time decay con Adam(estimación del momento adaptativo), esto es crucial durante el entrenamiento
, porque ayuda a los minimos de la funcion de costo en las epocas avanzadas. es decir que un learning rate constante hace que salte de un lado a otro sin estabilizarse,
generando una inestabilidad en la grafica de validación, al aplicar inverse time decay frenamos ese comportamiento, estabilizando la curva y permitiendo que el accuracy suba de forma mas suave.


In [ ]:
def compile_and_fit(model, train_dataset,val_dataset, initial_learning_rate:float=0.001, decay_steps:int=500, decay_rate:float=0.5)->tuple[tf.keras.callbacks.History]:
    """Compila y entrena un modelo de keras utilizando datasets estructurados.
    Args:
        model (tf.keras.Model): Instancia del modelo a entrenar
        train_dataset (tf.keras.datasets): Dataset de entrenamiento optimizado.
        val_dataset (tf.keras.datasets): Dataset de validación.
        initial_learning_rate (float): Tasa de aprendisaje incial para Adam.
        decay_steps: Número de pasos (batches) base para el decaimiento.
        decay_rate: Factor de decaimiento inverso.
    Returns:
        _type_: _description_
    """
    # implementacion de InverseTimeDecay    
    lr_schudle= tf.keras.optimizers.schedules.InverseTimeDecay(initial_learning_rate=initial_learning_rate,
                                                               decay_steps=decay_steps,
                                                               decay_rate=decay_rate,
                                                               staircase=False)
    
    optimizer= tf.keras.optimizers.Adam(learning_rate=lr_schudle)

    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=optimizer,
        metrics=['accuracy'],
    )
    histories= model.fit(
        train_dataset,
        epochs=EPOCHS,
        validation_data=val_dataset,
        callbacks=[callbacks_ear_stopp, callbacks_TensorBoard[model.name]],
        verbose=0,
    )
    return model, histories 

# Región de Entrenamiento MLP y CNN

In [ ]:
# modelo base mlp
fit_model_mlp, histories_mlp= compile_and_fit(model_mlp, train_dataset, val_dataset)

In [ ]:
# model red_C full regularization
fit_complex_model, histories_cnn =compile_and_fit(complex_model, train_dataset,val_dataset)

In [ ]:
# crear una funcion Gap
def gap(model:tf.keras.Model,X_eval_data:tf.data.Dataset | np.ndarray, val_data: tf.data.Dataset | np.ndarray):
    """Calcula la brecha de precisión usando
    de manera automatica el nombre nativo del modelo keras.

    Args:
        model (tf.keras.Model): Modelo de tensorflow/keras entrenado a evaluar.
        X_eval_data (np.ndarray): Conjunto de datos de evaluación.
        val_data (np.ndarray): Conjunto de datos de validación.

    Returns:
        _type_: _description_
    """    
    eval_loss, eval_acc= model.evaluate(X_eval_data, verbose=0)
    val_loss, val_acc= model.evaluate(val_data, verbose=0)

    gap= eval_loss - val_acc

    if gap <= 0.015:
        message= f'La brecha es igual o menor a 1.5%, El modelo {model.name} generaliza correctamente.'
    else:
        message= f'!El modelo supera la brecha de 1.5%, ({gap:.3f}) en modelo {model.name}, posible sobreajuste.'
    print(f"""
    Modelo evaluado:     {model.name}
    Evaluación accuracy: {eval_acc:.3f}
    Val accuracy:        {val_acc:.3f}
    Val loss:            {val_loss:.3f}
    Brecha calculada:    {gap:.3f}
    Diagnóstico:         {message}
    """)
    return gap


In [ ]:
brecha_mlp= gap(model_mlp,test_dataset, val_dataset)

In [ ]:
brecha_cnn= gap(complex_model, test_dataset, val_dataset)

In [ ]:
y_predict=[]
y_verdaderos=[]

for img, label in test_dataset:
    predicciones= complex_model.predict(img, verbose=0)
    y_verdaderos.extend(label.numpy().tolist())
    y_predict.extend(np.argmax(predicciones, axis=1).tolist())
print(classification_report(y_verdaderos, y_predict, target_names=class_names, zero_division=0,))

# Procesamiento Real
- El siguiente experimento consiste en realizar predicciones con imagenes reales.

In [ ]:
# este la linea de codigo procesara las imagenes reales que se encuentran en el directorio dataReal con imagenes reales, bebe contener las 25 clases
# para procesar este codigo o indicara error de fuera de rango.

# obtenemos las clases que se encuentran en el dataset de la ruta de las imagenes.

print("Clases detectadas en tus carpetas:", class_names)

# Aqui se encuentra la ruta de la foto
ruta_foto_movil = (
    "../dataReal/A/foto_1.jpg"
)

try:
  if not os.path.exists(ruta_foto_movil):
    raise FileNotFoundError(f"No se encontró la imagen en: {ruta_foto_movil}")

  # se carga la imagen con los parametros que se entrenaron
  img = tf.keras.utils.load_img(
      ruta_foto_movil,
      color_mode="grayscale",
      target_size=(IMG_HEIGHT, IMG_WIDTH),
  )

  #  Se convierte en un arreglo y se expande a dimensiones para crear el lote.
  img_array = tf.keras.utils.img_to_array(img)
  img_array = tf.expand_dims(img_array, axis=0)

  # Se aplica la funcion de normalización
  img_array = tf.cast(img_array, tf.float32) / 255.0

  # Predecimos con algunos de los modelos creados por defecto se toma el modelo red_c
  predicciones= complex_model.predict(img_array, verbose=0)
  clase_predicha_idx = np.argmax(predicciones[0])
  confianza = np.max(predicciones[0]) * 100

  clase_final = class_names[clase_predicha_idx]

  logger.info(
      f"Predicción exitosa para {ruta_foto_movil}: {clase_final} ({confianza:.2f}%)"
  )
  print(
      f"-> El modelo '{complex_model.name}' predice que es la letra: {clase_final}"
      f" con un {confianza:.2f}% de confianza."
  )

except Exception as e:
  logger.error(f"Error al procesar la imagen del móvil: {e}")

# Métricas y Evaluación

#### Gap (Brecha)

In [ ]:
# grafico gap red mlp y red_cnn
plt.figure(figsize=(9,8))
epochs= range(1, len(histories_mlp.history['accuracy'])+1)
plt.plot(epochs,histories_mlp.history['accuracy'],label='Train_accuracy',color='blue',linestyle='-')
plt.plot(epochs,histories_mlp.history['val_accuracy'],label='Val_accuracy',color='red',linestyle='--')
plt.fill_between(epochs,histories_mlp.history['accuracy'],histories_mlp.history['val_accuracy'],color='red',alpha=0.15,label=f'Brecha {brecha_mlp:.3f}',)
plt.title('Gap, evolucion del Overfitting Train vs Validacion',fontsize=12, fontweight='bold')
plt.xlabel('epocas',fontsize=12,fontweight='bold')
plt.ylabel('accuracy',fontsize=12,fontweight='bold')
plt.show()

In [ ]:
plt.figure(figsize=(9,8))
epochs= range(1, len(histories_cnn.history['accuracy'])+1)
plt.plot(epochs,histories_cnn.history['accuracy'],label='Train_accuracy',color='blue',linestyle='-')
plt.plot(epochs,histories_cnn.history['val_accuracy'],label='Val_accuracy',color='red',linestyle='--')
plt.fill_between(epochs,histories_cnn.history['accuracy'],histories_cnn.history['val_accuracy'],color='red',alpha=0.15,label=f'Brecha {brecha_cnn:.3f}',)
plt.title('Gap, evolucion del Overfitting Train vs Validacion',fontsize=12, fontweight='bold')
plt.xlabel('epocas',fontsize=12,fontweight='bold')
plt.ylabel('accuracy',fontsize=12,fontweight='bold')
plt.legend(loc='lower right')
plt.show()

### Matriz de confusión (cm) y analís de curva de aprendisaje

In [ ]:
def confusion_matrix(
    model: tf.keras.Model, dataset: tf.data.Dataset, titulo: str = "Matriz de confusión - Sign MNIST Test",
    normalize: str | None = None,
    figsize: tuple[int, int] = (10, 9),
):
  """Calcula y grafica la matriz de confusión sobre test_dataset.

  Args:
      model: Modelo entrenado de Keras (CNN o MLP).
      dataset: tf.data.Dataset de prueba (ej. test_dataset).
      titulo: Título del gráfico.
      normalize: None para conteos absolutos, o 'true' para proporción por
        clase real.
      figsize: Tamaño de la figura en pulgadas.
  """
  y_true_list = []
  for batch in dataset:
    if isinstance(batch, (tuple, list)) and len(batch) >= 2:
      y_batch = batch[1]
      y_true_list.append(y_batch.numpy() if hasattr(y_batch, "numpy") else y_batch)
    else:
      raise TypeError(
          "Se esperaba que cada elemento de 'dataset' fuera una tupla/batch (x, y). "
          "Si tienes arreglos (X_test, y_test), pásalos por separado."
      )

  y_true = np.concatenate(y_true_list, axis=0)
  if y_true.ndim > 1:
    y_true = np.argmax(y_true, axis=1)

  y_pred_probs = model.predict(dataset, verbose=0)
  y_pred = np.argmax(y_pred_probs, axis=1)

  cm = sk_confusion_matrix(y_true, y_pred, normalize=normalize)

  fig, ax = plt.subplots(figsize=figsize)
  disp = ConfusionMatrixDisplay(confusion_matrix=cm)
  disp.plot(ax=ax, cmap="Blues", values_format=".2f" if normalize else "d")
  ax.set_title(titulo)
  plt.tight_layout()
  plt.show()
  return cm

In [ ]:
#  Matriz con conteos absolutos (CNN)
cm_cnn = confusion_matrix(
    fit_complex_model,
    test_dataset,
    titulo="Matriz de confusión (Conteos) - CNN Sign MNIST Test",
    normalize=None,
)

# Matriz normalizada por fila (para ver la tasa de acierto/confusión por clase)
cm_cnn_norm = confusion_matrix(
    fit_complex_model,
    test_dataset,
    titulo="Matriz de confusión (Normalizada) - CNN Sign MNIST Test",
    normalize="true",
)

# 3. Comparación con el baseline MLP
cm_mlp = confusion_matrix(
    fit_model_mlp,
    test_dataset,
    titulo="Matriz de confusión (Conteos) - MLP Sign MNIST Test",
    normalize=None,
)

## Implementacion de Grad-CAM
- Mediante el *Grad-CAM* es posible observar que regiones de la imagen (28x28 escala de grises) se concentra la activacion convolucional al clasificar una sena.
- Podemos observar que las posiciones de los dedos son similares entre clases y las variaciones de iluminacion y contraste se reflejan en la imagen, como los bordes y sombras de las manos mas sutil, esto dificulta al modelo en la distincion entre clases.
- Al incorporar Data Augmentation de contraste, exponemos a la red a distintas intensidades de contraste durante el entrenamiento, ayudando a la red a aprender representaciones mas robustas y a generalizar mejor ante estos cambios de luz en las posiciones de las manos parecidas.

In [ ]:
def get_last_conv_layer_name(model: tf.keras.Model) -> str:
  """Busca automáticamente el nombre de la última capa Conv2D del modelo."""
  for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
      return layer.name
  raise ValueError("No se encontró ninguna capa Conv2D en el modelo.")


def compute_gradcam_heatmap(
    model: tf.keras.Model,
    img_array: np.ndarray,
    last_conv_layer_name: str | None = None,
    pred_index: int | None = None,
) -> tuple[np.ndarray, int, float]:
  """Calcula el mapa de calor Grad-CAM para una sola imagen con forma (28, 28, 1)."""
  if last_conv_layer_name is None:
    last_conv_layer_name = get_last_conv_layer_name(model)

  img_batch = np.expand_dims(img_array, axis=0).astype(np.float32)

  # Asegurar que el modelo esté construido con pesos inicializados/cargados
  _ = model(img_batch, training=False)

  # Trazar grafo simbólico desde una entrada explícita para evitar el error de Sequential en Keras 3
  inputs = tf.keras.Input(shape=img_array.shape, dtype=tf.float32)
  x = inputs
  conv_output_tensor = None

  for layer in model.layers:
    # Evitar recrear una capa Input si ya viene dentro de model.layers
    if isinstance(layer, tf.keras.layers.InputLayer):
      continue
    x = layer(x, training=False) if "training" in layer.call.__code__.co_varnames else layer(x)
    if layer.name == last_conv_layer_name:
      conv_output_tensor = x

  if conv_output_tensor is None:
    raise ValueError(f"No se encontró la capa '{last_conv_layer_name}' al trazar el modelo.")

  grad_model = tf.keras.Model(
      inputs=inputs,
      outputs=[conv_output_tensor, x],
  )

  with tf.GradientTape() as tape:
    conv_outputs, predictions = grad_model(img_batch, training=False)
    if pred_index is None:
      pred_index = int(tf.argmax(predictions[0]))
    class_channel = predictions[:, pred_index]

  grads = tape.gradient(class_channel, conv_outputs)
  pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

  conv_outputs = conv_outputs[0]
  heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
  heatmap = tf.squeeze(heatmap, axis=-1)

  heatmap = tf.maximum(heatmap, 0)
  max_val = tf.math.reduce_max(heatmap)
  if max_val > 0:
    heatmap = heatmap / max_val

  confidence = float(predictions[0, pred_index].numpy())
  return heatmap.numpy(), pred_index, confidence


def plot_gradcam_comparison(
    model: tf.keras.Model,
    img_28x28x1: np.ndarray,
    true_label: int | None = None,
    class_names: list[str] | None = None,
    last_conv_layer_name: str | None = None,
    save_path: str | None = None,
):
  """Muestra la imagen original, el mapa Grad-CAM y la superposición."""
  heatmap, pred_idx, conf = compute_gradcam_heatmap(
      model, img_28x28x1, last_conv_layer_name=last_conv_layer_name
  )

  # Reescalar heatmap al tamaño de la imagen (28, 28)
  h, w = img_28x28x1.shape[0], img_28x28x1.shape[1]
  heatmap_resized = tf.image.resize(
      heatmap[..., np.newaxis], (h, w), method="bilinear"
  ).numpy()[..., 0]

  # Imagen base en escala de grises -> RGB para superponer colormap
  base_gray = np.squeeze(img_28x28x1).astype(np.float32)
  if base_gray.max() > 1.0:
    base_gray = base_gray / 255.0
  base_rgb = np.stack([base_gray] * 3, axis=-1)

  cmap = plt.get_cmap("jet")
  heatmap_rgb = cmap(heatmap_resized)[..., :3]
  overlay = np.clip(0.55 * base_rgb + 0.45 * heatmap_rgb, 0.0, 1.0)

  pred_name = class_names[pred_idx] if class_names else str(pred_idx)
  true_str = ""
  if true_label is not None:
    true_name = class_names[true_label] if class_names else str(true_label)
    true_str = f" | Real: {true_name}"

  fig, axes = plt.subplots(1, 3, figsize=(10, 3.3))

  axes[0].imshow(base_gray, cmap="gray")
  axes[0].set_title("Entrada (28x28)")
  axes[0].axis("off")

  axes[1].imshow(heatmap_resized, cmap="jet", vmin=0.0, vmax=1.0)
  axes[1].set_title("Grad-CAM")
  axes[1].axis("off")

  axes[2].imshow(overlay)
  axes[2].set_title(f"Pred: {pred_name} ({conf:.1%}){true_str}")
  axes[2].axis("off")

  plt.tight_layout()
  if save_path:
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
  plt.show()

In [ ]:
for img_batch, label_batch in test_dataset.take(1):
  sample_img = img_batch[0].numpy()
  sample_label = int(label_batch[0].numpy())
  plot_gradcam_comparison(
      model=complex_model,
      img_28x28x1=sample_img,
      true_label=sample_label,
      class_names=class_names,
      save_path="gradcam_ejemplo.png",
  )
  break

### Curva de roc (Caracteristica Operativa del Receptor)


In [ ]:
def _get_y_true_and_scores(
    model: tf.keras.Model, dataset: tf.data.Dataset
) -> tuple[np.ndarray, np.ndarray]:
  """Obtiene etiquetas reales (one-hot) y probabilidades predichas (Softmax)."""
  y_true = np.concatenate([y.numpy() for _, y in dataset], axis=0)
  y_score = model.predict(dataset, verbose=0)
  num_classes = y_score.shape[1]

  if y_true.ndim == 1:
    y_true = tf.keras.utils.to_categorical(
        y_true.astype(int), num_classes=num_classes
    )

  return y_true, y_score


def plot_roc_auc_mnist(
    model: tf.keras.Model,
    dataset: tf.data.Dataset,
    titulo: str = 'Curva ROC-AUC - Sign MNIST Test',
) -> dict[str, float]:
  """Calcula y grafica la curva ROC (Micro-AUC y Macro-AUC OvR) en Sign MNIST test."""
  y_true, y_score = _get_y_true_and_scores(model, dataset)
  active = np.where(y_true.sum(axis=0) > 0)[0]

  y_true_act = y_true[:, active]
  y_score_act = y_score[:, active]

  micro_auc = roc_auc_score(
      y_true_act, y_score_act, multi_class='ovr', average='micro'
  )
  macro_auc = roc_auc_score(
      y_true_act, y_score_act, multi_class='ovr', average='macro'
  )

  fpr_micro, tpr_micro, _ = roc_curve(y_true_act.ravel(), y_score_act.ravel())

  plt.figure(figsize=(6.5, 5))
  plt.plot(
      fpr_micro,
      tpr_micro,
      linewidth=2,
      label=f'Micro ROC-AUC = {micro_auc:.4f} | Macro ROC-AUC = {macro_auc:.4f}',
  )
  plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Azar (0.50)')
  plt.xlabel('Tasa de Falsos Positivos (FPR)')
  plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
  plt.title(titulo)
  plt.legend(loc='lower right')
  plt.grid(True, alpha=0.3)
  plt.tight_layout()
  plt.show()

  return {'micro_auc': micro_auc, 'macro_auc': macro_auc}

In [ ]:
# Evaluación en Sign MNIST test (sin imágenes reales)
auc_cnn_mnist = plot_roc_auc_mnist(
    fit_complex_model, test_dataset, 'ROC-AUC CNN - Sign MNIST Test'
)
auc_mlp_mnist = plot_roc_auc_mnist(
    fit_model_mlp, test_dataset, 'ROC-AUC MLP - Sign MNIST Test'
)

### Curva de aprendisaje MLP y CNN (Accuracy, val_accuracy, loss, val_loss)

In [ ]:
# MLP
plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.plot(histories_mlp.history['accuracy'], label='accuracy')
plt.plot(histories_mlp.history['val_accuracy'], label='val accuracy')
plt.title('accuracy validacion')
plt.xlabel('epocas')
plt.ylabel('accuracy')
plt.legend(loc='lower right')

plt.subplot(1,2,2)
plt.plot(histories_mlp.history['loss'], label='loss')
plt.plot(histories_mlp.history['val_loss'], label='val_loss')
plt.title('perdida validacion')
plt.xlabel('epocas')
plt.ylabel('perdida')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# red CNN
plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.plot(histories_cnn.history['accuracy'], label='accuracy')
plt.plot(histories_cnn.history['val_accuracy'], label='val accuracy')
plt.title('Accuracy validacion')
plt.xlabel('epocas')
plt.ylabel('accuracy')
plt.legend(loc='lower right')

plt.subplot(1,2,2)
plt.plot(histories_cnn.history['loss'], label='loss')
plt.plot(histories_cnn.history['val_loss'], label='val_loss')
plt.title('perdida validacion')
plt.xlabel('epocas')
plt.ylabel('perdida')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

### Predicciones

In [ ]:
# Obtener predicciones
predicciones = complex_model.predict(X_test)
clases_predichas = np.argmax(predicciones, axis=1)
# Diccionario inverso: clase -> letra
letras = {valor: clave for clave, valor in abecedario.items()}
# Seleccionar 10 imágenes aleatorias sin repetir
indices_aleatorios = np.random.choice(
    len(X_test), 10, replace=False
)
# Crear visualización
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()
for i, idx in enumerate(indices_aleatorios):
    # Imagen
    img = tf.reshape(X_test[idx], (28, 28))
    # Etiqueta real y predicción
    real_label = int(label_test[idx])
    pred_label = int(clases_predichas[idx])
    # Mostrar imagen
    axes[i].imshow(img, cmap='gray')
    # Verde si acertó, rojo si se equivocó
    color = 'green' if real_label == pred_label else 'red'
    axes[i].set_title(
        f"Real: {letras[real_label]}\n"
        f"Predicción: {letras[pred_label]}",
        color=color,
        fontsize=12,
        fontweight='bold'
    )
    axes[i].axis('off')
plt.suptitle(
    "Imágenes de prueba: valores reales vs. predichos",
    fontsize=16,
    fontweight='bold'
)
plt.tight_layout()
plt.show()

# Anexo
- No tiene que ver con el proyecto, solo es para reforzar el porque seleccionamos he_normal.

In [ ]:
# guardar modelos formato .keras
model_dir= '../model'
os.makedirs(model_dir, exist_ok=True)
model=[model_mlp, complex_model]
for mod in model:
    nombre_archivo= f'{mod.name}.keras'
    ruta_completa= os.path.join(model_dir, nombre_archivo)
    mod.save(ruta_completa)
    print(f'Modelo {mod.name} guardado correctamente en ruta {ruta_completa}')

In [ ]:
def build_model_cnn_init(
    n_filters: np.ndarray,
    kernel_initializer="he_normal",
    input_shape=(28, 28, 1),
    num_class: int = 25,
    model_name: str = "model_cnn",
) -> tf.keras.Model:
  """Variante de build_model_cnn con inicializador configurable en capas con ReLU."""
  model = tf.keras.models.Sequential(
      [
          tf.keras.layers.Input(shape=input_shape, name="Input_layer"),
          tf.keras.layers.RandomRotation(factor=0.03, name="aug_rotation"),
          tf.keras.layers.RandomZoom(
              height_factor=(-0.05, 0.05),
              width_factor=(-0.05, 0.05),
              name="aug_zoom",
          ),
          tf.keras.layers.RandomContrast(factor=0.1, name="aug_contrast"),
          tf.keras.layers.Conv2D(
              filters=n_filters[0],
              kernel_size=(3, 3),
              activation="relu",
              kernel_initializer=kernel_initializer,
              name="layer_conv_1",
          ),
          tf.keras.layers.BatchNormalization(name="batch_normalization"),
          tf.keras.layers.MaxPool2D((2, 2), name="Maxpool_1"),
          tf.keras.layers.Flatten(name="flatten"),
          tf.keras.layers.Dropout(0.2, name="layer_Dropout_0.2"),
          tf.keras.layers.Dense(
              128,
              activation="relu",
              kernel_initializer=kernel_initializer,
              name="layer_Dense_1",
          ),
          tf.keras.layers.Dense(
              num_class, activation="softmax", name="layer_output"
          ),
      ],
      name=model_name,
  )
  return model

In [ ]:
experimentos_cnn = {
    "Con he_normal": {
        "init": "he_normal",
        "model_name": "cnn_con_he_normal",
    },
    "Sin he_normal (Glorot/Xavier)": {
        "init": "glorot_uniform",
        "model_name": "cnn_sin_he_normal_glorot",
    },
    "Sin he_normal (Normal std=0.1)": {
        "init": tf.keras.initializers.RandomNormal(
            mean=0.0, stddev=0.1, seed=42
        ),
        "model_name": "cnn_sin_he_normal_rnormal",
    },
}

histories_cnn = {}
n_filters= np.array([64])
for label, cfg in experimentos_cnn.items():
  m_name= cfg['model_name']
  if m_name not in callbacks_TensorBoard:
    log_dir=(f'log/{m_name}_{datetime.datetime.now().strftime('"%d-%m-%y,%H-%M-%S"')}')
    callbacks_TensorBoard[m_name]= tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

histories_cnn={}

for label, cfg in experimentos_cnn.items():
  tf.keras.utils.set_random_seed(42)
  cnn_model = build_model_cnn_init(
      n_filters=n_filters,
      kernel_initializer=cfg["init"],
      model_name=cfg["model_name"],
  )
  _, hist = compile_and_fit(
      cnn_model,
      train_dataset=train_dataset,
      val_dataset=val_dataset,
  )
  histories_cnn[label] = hist.history

# Gráficos comparativos (Loss y Accuracy)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

for label, h in histories_cnn.items():
  axes[0].plot(h["loss"], label=f"{label} (train)")
  axes[0].plot(h["val_loss"], linestyle="--", label=f"{label} (val)")

axes[0].set_title("CNN - Evolución de la pérdida (Loss)")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

for label, h in histories_cnn.items():
  axes[1].plot(h["accuracy"], label=f"{label} (train)")
  axes[1].plot(h["val_accuracy"], linestyle="--", label=f"{label} (val)")

axes[1].set_title("CNN - Evolución de la precisión (Accuracy)")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Accuracy")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("comparacion_he_normal_cnn.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# visualizacion mediante tensorboard
%load_ext tensorboard
%tensorboard --logdir log/